In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt


con = duckdb.connect("healthcare.duckdb")


In [8]:
con.close()

## target data

- atc3 : hosp, (sick)
    - atc3_region_facil <- atc2cd column 생성
- atc4 : hosp, sick
    - atc4_region_inst -> atc4_region_facil
    - atc4_sick
- regional medical facility distribution data : 2021,2022
    - region_facil_stat
- atc master : atc_master


### if possible
- mefi : sick

In [ ]:
# rename column name 

## ref - 3단계
# atcStep3Cd	atcStep3CdNm	diagYm	insupTpCd	msupUseAmt	st3SickSym	st3SickSymNm	totUseQty
# A02A	ANTACIDS	202201	4	20258	AE10	1형 당뇨병	549483

# 4단계ATC별상병별사용량목록조회 2020-2022/09_getAtcStp4SickList_2020.parquet
# 202001	1형 당뇨병	A01AD	4	16	17239	AE10	Other agents for local oral treatme
# diagYm - st3SickSymNm - atcStep4Cd- insupTpCd - msupUseAmt - totUseQty - st3SickSym - atcStep4CdNm
  
# 성분별상병별사용량목록조회 2020-2022/12_getCmpnSickList_2022.parquet
# 202201	1형 당뇨병	100701ACH	acebrophylline	4	424	86148	AE10
# diagYm - st3SickSymNm - gnlNmCd  - gnlNmCdNm - insupTpCd - msupUseAmt - totUseQty - st3SickSym

# concat same  data by diagYm

In [3]:
# con.execute("show tables like '%region%';").df()

con.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    -- WHERE table_name LIKE '%region_%';

""").df()

,table_name
0,diag_atc3_compare
1,diag_region_atc3
2,diag_sick_atc3
3,atc2_monthly
4,atc3_region_facil
5,atc3_sick
6,atc3_sick_base
7,atc4_region_facil
8,atc4_sick
9,atc_master


In [6]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

con.execute("""
CREATE OR REPLACE VIEW atc3_sick AS
SELECT 
* exclude(totUseQty,msupUseAmt) ,
msupUseAmt as totUseQty,
totUseQty as msupUseAmt,
FROM read_parquet(
    'parquet/atc3_sick/new_*.parquet',
    union_by_name = true
);
""")
con.execute("select * from atc3_sick limit 1;").df()

,atcStep3Cd,atcStep3CdNm,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt
0,A12C,OTHER MINERAL SUPPLEMENTS,202001,5,AE11,2형 당뇨병,50,70031


In [6]:
con.execute("select left(DIAGYM, 4) as year, sum(msupUseAmt)/1e12 as use_amt_tril from atc3_sick  group by 1 ;").df()

,year,use_amt_tril
0,2021,23.419226
1,2020,21.842129
2,2022,23.670950


In [4]:
con.execute("""
CREATE OR REPLACE VIEW atc4_sick AS
SELECT  
    col_1::VARCHAR AS diagYm,
    col_2::VARCHAR AS st3SickSymNm,
    col_3::VARCHAR AS atcStep4Cd,
    col_4::VARCHAR AS insupTpCd,
    TRY_CAST(col_5 AS DOUBLE) AS totUseQty,
    TRY_CAST(col_6 AS DOUBLE) AS msupUseAmt,
    col_7::VARCHAR AS st3SickSym,
    col_8::VARCHAR AS atcStep4CdNm
FROM read_parquet(
    'parquet/atc4_sick/new_*.parquet',
    union_by_name = true
)
;
""")
con.execute("select * from atc4_sick limit 1;").df()

,diagYm,st3SickSymNm,atcStep4Cd,insupTpCd,totUseQty,msupUseAmt,st3SickSym,atcStep4CdNm
0,202001,1형 당뇨병,A02BX,4,24224.0,3937362.0,AE10,Other drugs for peptic ulcer and gastro-oesoph...


In [4]:
con.execute("""
CREATE OR REPLACE VIEW atc4_region_facil AS
SELECT  
    col_1::VARCHAR AS diagYm,
        LEFT(col_2::VARCHAR,4) AS atcStep3Cd,
        col_2::VARCHAR AS atcStep4Cd,
        col_3::VARCHAR AS regionStep2Cd,
        col_4::VARCHAR AS regionStep1CdNm,
        col_5::VARCHAR AS regionStep2CdNm,
        col_6::VARCHAR AS insupTpCd,
        col_7::VARCHAR AS regionStep1Cd,
        TRY_CAST(col_8 AS DOUBLE) AS totUseQty,
        TRY_CAST(col_9 AS DOUBLE) AS msupUseAmt,
        col_10::VARCHAR AS medInstType,
        col_11::VARCHAR AS atcStep4CdNm  
FROM read_parquet(
    'parquet/atc4_hosp/new*.parquet',
    union_by_name = true
)
;
""")
con.execute("select * from atc4_region_facil limit 1;").df()

,diagYm,atcStep3Cd,atcStep4Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep4CdNm
0,202001,C10A,C10AA,240003,광주,광주서구,4,24,211.0,135762.0,의원,HMG CoA reductase inhibitors


In [ ]:
# con.execute("""
# CREATE OR REPLACE VIEW atc4_region_facil_tmp AS
# SELECT  
#     col_1::VARCHAR AS diagYm,
#         LEFT(col_2::VARCHAR,4) AS atcStep3Cd,
#         col_2::VARCHAR AS atcStep4Cd,
#         col_3::VARCHAR AS regionStep2Cd,
#         col_4::VARCHAR AS regionStep1CdNm,
#         col_5::VARCHAR AS regionStep2CdNm,
#         col_6::VARCHAR AS insupTpCd,
#         col_7::VARCHAR AS regionStep1Cd,
#         TRY_CAST(col_8 AS DOUBLE) AS totUseQty,
#         TRY_CAST(col_9 AS DOUBLE) AS msupUseAmt,
#         col_10::VARCHAR AS medInstType,
#         col_11::VARCHAR AS atcStep4CdNm  
# FROM read_parquet(
#     'parquet/atc4_hosp/new*.parquet',
#     union_by_name = true
# )
# ;
# """)
# con.execute("select * from atc4_region_facil_tmp where msupUseAmt = 32844.0 and regionStep2Cd = 312001	and atcStep4Cd = 'N05BE' limit 1;").df()

,diagYm,atcStep3Cd,atcStep4Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep4CdNm
0,202001,N05B,N05BE,312001,경기,용인기흥구,4,31,286.0,32844.0,종합병원,Azaspirodecanedione derivatives


In [ ]:
# con.execute("""
# CREATE OR REPLACE VIEW atc4_region_facil_tmp AS
# SELECT  
#     col_1::VARCHAR AS diagYm,
#         LEFT(col_2::VARCHAR,4) AS atcStep3Cd,
#         col_2::VARCHAR AS atcStep4Cd,
#         col_3::VARCHAR AS regionStep2Cd,
#         col_4::VARCHAR AS regionStep1CdNm,
#         col_5::VARCHAR AS regionStep2CdNm,
#         col_6::VARCHAR AS insupTpCd,
#         col_7::VARCHAR AS regionStep1Cd,
#         TRY_CAST(col_8 AS DOUBLE) AS totUseQty,
#         TRY_CAST(col_9 AS DOUBLE) AS msupUseAmt,
#         col_10::VARCHAR AS medInstType,
#         col_11::VARCHAR AS atcStep4CdNm  
# FROM read_parquet(
#     'parquet/atc4_hosp/08*.parquet',
#     union_by_name = true
# )
# ;
# """)
# con.execute("select * from atc4_region_facil_tmp where msupUseAmt = 32844.0 and regionStep2Cd = 312001	and atcStep4Cd = 'N05BE' limit 1;").df()

,diagYm,atcStep3Cd,atcStep4Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep4CdNm
0,202001,N05B,N05BE,312001,경기,용인기흥구,4,11,286.0,32844.0,종합병원,Azaspirodecanedione derivatives


In [3]:
con.execute("""
CREATE OR REPLACE VIEW atc3_region_facil  as
SELECT
    col_1::VARCHAR AS diagYm,
    LEFT(col_2::VARCHAR, 3) AS atcStep2Cd,
    col_2::VARCHAR AS atcStep3Cd,
    col_3::VARCHAR AS regionStep2Cd,
    col_4::VARCHAR AS regionStep1CdNm,
    col_5::VARCHAR AS regionStep2CdNm,
    col_6::VARCHAR AS insupTpCd,
    col_7::VARCHAR AS regionStep1Cd,
    TRY_CAST(col_8 AS DOUBLE) AS totUseQty,
    TRY_CAST(col_9 AS DOUBLE) AS msupUseAmt,
    col_10::VARCHAR AS medInstType,
    col_11::VARCHAR AS atcStep3CdNm    
     
    FROM read_parquet(
    'parquet/atc3_hosp/new*.parquet',
    union_by_name = true
) 
; """).df()

con.execute("select * from atc3_region_facil limit 1;").df()

,diagYm,atcStep2Cd,atcStep3Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep3CdNm
0,202002,C10,C10A,110004,서울,관악구,5,11,72956.0,36486550.0,약국,"LIPID MODIFYING AGENTS, PLAIN"


In [12]:
con.execute("""
 CREATE OR REPLACE VIEW region_facil_stat AS
SELECT
  diagYm,
  sidoNm, -- join with region code table on regionStep1CdNm
  TRY_CAST(REPLACE(TRIM(population), ',', '') AS BIGINT) AS population,
  TRY_CAST(REPLACE(TRIM(pharm), ',', '') AS BIGINT) AS pharm,
  TRY_CAST(REPLACE(TRIM(advGenHosp), ',', '') AS BIGINT) AS advGenHosp,
  TRY_CAST(REPLACE(TRIM(genHosp), ',', '') AS BIGINT) AS genHosp,
  TRY_CAST(REPLACE(TRIM(hosp), ',', '') AS BIGINT) AS hosp,
  TRY_CAST(REPLACE(TRIM(longHosp), ',', '') AS BIGINT) AS longHosp,
  TRY_CAST(REPLACE(TRIM(clinic), ',', '') AS BIGINT) AS clinic,
  TRY_CAST(REPLACE(TRIM(dentalClinic), ',', '') AS BIGINT) AS dentalClinic,
  TRY_CAST(REPLACE(TRIM(orientalClinic), ',', '') AS BIGINT) AS orientalClinic
FROM read_csv(
  'raw_data/region_medi_facility_2022.tsv', 
  delim='\t',
  all_varchar=True -- 모든 컬럼을 일단 문자열로 안전하게 읽어옴
);
  ; """)

con.execute("select * from region_facil_stat limit 1;").df()

,diagYm,sidoNm,population,pharm,advGenHosp,genHosp,hosp,longHosp,clinic,dentalClinic,orientalClinic
0,2021,서울,9509458,5831,13,44,236,105,10323,4903,3662


In [13]:
con.execute("""
 CREATE OR REPLACE VIEW atc_master AS
SELECT
*
FROM read_csv(
  'raw_data/atc_master.csv', 
  all_varchar=True 
);
""")
con.execute("""select * from atc_master limit 1;""").df()


,atc_code,atc_name,strength,uom,adm_r,note
0,A,ALIMENTARY TRACT AND METABOLISM,NA,NA,NA,NA


### 신규테이블 점검

In [11]:
import duckdb
import pandas as pd

con = duckdb.connect("healthcare.duckdb")

views_df = con.execute("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.views
    WHERE table_schema = 'main'
      AND table_name like 'atc%' OR table_name like 'region%'
    ORDER BY table_name
""").fetchdf()

display(views_df)

,table_schema,table_name
0,main,atc3_region_facil
1,main,atc3_sick
2,main,atc4_region_facil
3,main,atc4_sick
4,main,atc_master
5,main,region_facil_stat


In [9]:
con.execute("""DROP VIEW IF EXISTS atc4_region_facil_tmp;""")

In [12]:
VIEWS = views_df["table_name"].tolist()

row_counts = []

for view in VIEWS:
    n = con.execute(f"""
        SELECT COUNT(*)
        FROM "{view}"
    """).fetchone()[0]

    row_counts.append({
        "view": view,
        "row_count": n
    })

row_counts_df = pd.DataFrame(row_counts)

display(row_counts_df)

row_counts_dict = dict(
    zip(
        row_counts_df["view"],
        row_counts_df["row_count"]
    )
)

,view,row_count
0,atc3_region_facil,17114053
1,atc3_sick,14364479
2,atc4_region_facil,25748443
3,atc4_sick,20635895
4,atc_master,7536
5,region_facil_stat,34


In [7]:
try:

    nt = con.execute(f"""
            SELECT 
            count(distinct atcStep2Cd), 
            count(distinct atcStep3Cd)
            FROM atc3_region_facil 
        """).fetchone()
    
    print("atc3_region_facil / unique atcStep2Cd : " + str(nt[0]))
    print("atc3_region_facil / unique atcStep3Cd : " + str(nt[1]) )

    nt = con.execute(f"""
            SELECT 
                'ATC2CD_YM_GT24_CNT' AS metric, count(distinct atcStep2Cd)
            FROM (
                select atcstep2Cd, COUNT(*) from atc3_region_facil group by atcstep2Cd having count(DISTINCT DIAGYM) > 24
            )
            UNION ALL
            SELECT 
                'ATC3CD_GT24_CNT' AS metric, count(distinct atcStep3Cd)
            FROM (
                select atcstep3Cd, COUNT(*) from atc3_region_facil group by atcstep3Cd having count(DISTINCT DIAGYM) > 24
            ) 
        """).fetchall()
    
    print( nt)

    nt = con.execute(f"""
        SELECT 
        count(distinct atcStep3Cd), count(distinct atcStep4Cd)
        FROM atc4_region_facil 
    """).fetchone()

    print("atc4_region_facil / unique atcStep3Cd : " + str(nt[0]))
    print("atc4_region_facil / unique atcStep4Cd : " + str(nt[1]))

    nt = con.execute(f"""
                SELECT 
                    'ATC3CD_YM_GT24_CNT' AS metric, count(distinct atcStep3Cd)
                FROM (
                    select atcstep3Cd, COUNT(*) from atc4_region_facil group by atcstep3Cd having count(DISTINCT DIAGYM) > 24
                )
                UNION ALL
                SELECT 
                    'ATC4CD_YM_GT24_CNT' AS metric, count(distinct atcStep4Cd)
                FROM (
                    select atcstep4Cd, COUNT(*) from atc4_region_facil group by atcstep4Cd having count(DISTINCT DIAGYM) > 24
                ) 
            """).fetchall()
        
    print(nt)
    
except Exception as e:
    print(f"Error occurred: {e}")

atc3_region_facil / unique atcStep2Cd : 78
atc3_region_facil / unique atcStep3Cd : 170
[('ATC2CD_YM_GT24_CNT', 72), ('ATC3CD_GT24_CNT', 155)]
atc4_region_facil / unique atcStep3Cd : 164
atc4_region_facil / unique atcStep4Cd : 321
[('ATC3CD_YM_GT24_CNT', 145), ('ATC4CD_YM_GT24_CNT', 268)]


In [7]:
## SICK , REGION FACIL -> AMT DIFFERENCE

try :
    nt = con.execute(f"""
                SELECT 
                    'ATC3_REGION_FACIL' as table_name, left(diagYm, 4)::INTEGER AS year,
                    sum(msupUseAmt)/1e12 AS total_msupUseAmt,
                    sum(totUseQty) AS total_totUseQty
                FROM atc3_region_facil 
                WHERE insupTpCd IN ('4', '5', '7')
                GROUP BY 1,2
                UNION ALL
                SELECT 
                    'ATC3_SICK' as table_name, left(diagYm, 4)::INTEGER AS year,
                    sum(msupUseAmt)/1e12 AS total_msupUseAmt,
                    sum(totUseQty) AS total_totUseQty
                FROM atc3_sick
                WHERE insupTpCd IN ('4', '5', '7')
                GROUP BY 1,2

            """).df().sort_values(by=[ "table_name","year"], ascending=[True, True])
        
    display(nt)
except Exception as e:
    print(f"Error occurred: {e}") 

,table_name,year,total_msupUseAmt,total_totUseQty
2,ATC3_REGION_FACIL,2020,23.458417,5.295534e+10
0,ATC3_REGION_FACIL,2021,25.248421,5.543093e+10
1,ATC3_REGION_FACIL,2022,26.479837,6.363364e+10
5,ATC3_SICK,2020,15.693051,3.127194e+10
3,ATC3_SICK,2021,17.440599,3.434033e+10
4,ATC3_SICK,2022,18.062456,3.972653e+10


In [24]:
con.execute("""
with cte as (

    SELECT * exclude(totUseQty,msupUseAmt) ,
    msupUseAmt as totUseQty,
    totUseQty as msupUseAmt,
    FROM read_parquet(
        'parquet/mefi_sick/*.parquet',
        union_by_name = true
    )
)
select 
    'mefi_SICK' as table_name, 
    left(diagYm, 4)::INTEGER AS year,
    sum(msupUseAmt)/1e12 AS total_msupUseAmt
from 
    cte
group by 1,2
;
""").df()


,table_name,year,total_msupUseAmt
0,mefi_SICK,2020,37.845982
1,mefi_SICK,2021,41.256710
2,mefi_SICK,2022,42.157308


In [15]:
# ============================================================
# 3. VIEW 컬럼 구조
# ============================================================
VIEWS = ['atc3_sick', 'atc4_sick', 'atc3_region_facil', 'atc4_region_facil', 'region_facil_stat', 'atc_master']

def get_columns(view):
    return con.execute(f"""
        SELECT
            ordinal_position,
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'main'
          AND table_name = '{view}'
        ORDER BY ordinal_position
    """).fetchdf()


for view in VIEWS:

    print(f"\n{'=' * 80}")
    print(f"[{view}]")
    print("=" * 80)

    display(get_columns(view))


[atc3_sick]


,ordinal_position,column_name,data_type
0,1,atcStep3Cd,VARCHAR
1,2,atcStep3CdNm,VARCHAR
2,3,diagYm,VARCHAR
3,4,insupTpCd,VARCHAR
4,5,st3SickSym,VARCHAR
5,6,st3SickSymNm,VARCHAR
6,7,totUseQty,BIGINT
7,8,msupUseAmt,BIGINT



[atc4_sick]


,ordinal_position,column_name,data_type
0,1,diagYm,VARCHAR
1,2,st3SickSymNm,VARCHAR
2,3,atcStep4Cd,VARCHAR
3,4,insupTpCd,VARCHAR
4,5,totUseQty,DOUBLE
5,6,msupUseAmt,DOUBLE
6,7,st3SickSym,VARCHAR
7,8,atcStep4CdNm,VARCHAR



[atc3_region_facil]


...

,ordinal_position,column_name,data_type
0,1,diagYm,VARCHAR
1,2,atcStep3Cd,VARCHAR
2,3,atcStep4Cd,VARCHAR
3,4,regionStep2Cd,VARCHAR
4,5,regionStep1CdNm,VARCHAR
5,6,regionStep2CdNm,VARCHAR
6,7,insupTpCd,VARCHAR
7,8,regionStep1Cd,VARCHAR
8,9,totUseQty,DOUBLE
9,10,msupUseAmt,DOUBLE



[region_facil_stat]


,ordinal_position,column_name,data_type
0,1,diagYm,VARCHAR
1,2,sidoNm,VARCHAR
2,3,population,BIGINT
3,4,pharm,BIGINT
4,5,advGenHosp,BIGINT
5,6,genHosp,BIGINT
6,7,hosp,BIGINT
7,8,longHosp,BIGINT
8,9,clinic,BIGINT
9,10,dentalClinic,BIGINT



[atc_master]


,ordinal_position,column_name,data_type
0,1,atc_code,VARCHAR
1,2,atc_name,VARCHAR
2,3,strength,VARCHAR
3,4,uom,VARCHAR
4,5,adm_r,VARCHAR
5,6,note,VARCHAR


In [16]:
def get_first_10rows(view):
    return con.execute(f"""
        SELECT
        *
            -- diagYm, insupTpCd, st3SickSym, st3SickSymNm, totUseQty, msupUseAmt, 
            -- * EXCLUDE(diagYm, insupTpCd, st3SickSym, st3SickSymNm, totUseQty, msupUseAmt) 
        FROM '{view}'
        limit 3
    """).fetchdf()

for view in VIEWS:

    print(f"\n{'=' * 80}")
    print(f"[{view}]")
    print("=" * 80)

    display(get_first_10rows(view))


[atc3_region_facil]


,diagYm,atcStep2Cd,atcStep3Cd,regionStep2Cd,regionStep1CdNm,regionStep2CdNm,insupTpCd,regionStep1Cd,totUseQty,msupUseAmt,medInstType,atcStep3CdNm
0,202002,C01,C01B,310800,경기,의정부시,4,21,58.0,31262.0,병원,"ANTIARRHYTHMICS, CLASS I AND III"
1,202002,S01,S01E,311000,경기,구리시,4,31,1.0,147.0,의원,ANTIGLAUCOMA PREPARATIONS AND MIOTICS
2,202002,R01,R01B,320700,강원,삼척시,5,11,25.0,2639.0,종합병원,NASAL DECONGESTANTS FOR SYSTEMIC USE



[atc3_sick]


,atcStep3Cd,atcStep3CdNm,diagYm,insupTpCd,st3SickSym,st3SickSymNm,totUseQty,msupUseAmt
0,A01A,STOMATOLOGICAL PREPARATIONS,202001,4,AE10,1형 당뇨병,16,17239
1,A01A,STOMATOLOGICAL PREPARATIONS,202001,5,AE10,1형 당뇨병,4,5000
2,A02A,ANTACIDS,202001,4,AE10,1형 당뇨병,21601,611090



[atc4_region_facil]


...

,diagYm,st3SickSymNm,atcStep4Cd,insupTpCd,totUseQty,msupUseAmt,st3SickSym,atcStep4CdNm
0,202001,1형 당뇨병,A01AD,4,16.0,17239.0,AE10,Other agents for local oral treatment
1,202001,1형 당뇨병,A01AD,5,4.0,5000.0,AE10,Other agents for local oral treatment
2,202001,1형 당뇨병,A02AA,4,16697.0,309538.0,AE10,Magnesium compounds



[atc_master]


,atc_code,atc_name,strength,uom,adm_r,note
0,A,ALIMENTARY TRACT AND METABOLISM,NA,NA,NA,NA
1,A01,STOMATOLOGICAL PREPARATIONS,NA,NA,NA,NA
2,A01A,STOMATOLOGICAL PREPARATIONS,NA,NA,NA,NA



[region_facil_stat]


,diagYm,sidoNm,population,pharm,advGenHosp,genHosp,hosp,longHosp,clinic,dentalClinic,orientalClinic
0,2021,서울특별시,9509458,5831,13,44,236,105,10323,4903,3662
1,2021,부산광역시,3392361,1718,4,27,145,154,2682,1353,1133
2,2021,대구광역시,2418754,1405,5,13,94,69,2049,955,901


## qc
- null
- numeric : zero, negative
- datetime
- duplicate

In [17]:



def null_quality(view_name):

    columns = get_columns(view_name)

    total_rows = con.execute(
        f'SELECT COUNT(*) FROM "{view_name}"'
    ).fetchone()[0]

    expressions = []

    for _, row in columns.iterrows():

        col = row["column_name"]
        dtype = str(row["data_type"]).upper()

        # column identifier 안전 처리
        qcol = '"' + col.replace('"', '""') + '"'

        # NULL
        expressions.append(f"""
            COUNT(*) FILTER (
                WHERE {qcol} IS NULL
            ) AS "{col}__null"
        """)

        # 문자열만 빈 문자열 검사
        if any(
            x in dtype
            for x in ["VARCHAR", "TEXT", "STRING"]
        ):
            expressions.append(f"""
                COUNT(*) FILTER (
                    WHERE {qcol} IS NOT NULL
                      AND TRIM(CAST({qcol} AS VARCHAR)) = ''
                ) AS "{col}__empty"
            """)

    sql = f"""
        SELECT
            {','.join(expressions)}
        FROM "{view_name}"
    """

    raw = con.execute(sql).fetchdf()

    results = []

    for _, row in columns.iterrows():

        col = row["column_name"]
        dtype = row["data_type"]

        null_count = int(raw.iloc[0][f"{col}__null"])

        empty_count = 0

        if any(
            x in str(dtype).upper()
            for x in ["VARCHAR", "TEXT", "STRING"]
        ):
            empty_count = int(
                raw.iloc[0][f"{col}__empty"]
            )

        results.append({
            "view": view_name,
            "column": col,
            "data_type": dtype,
            "null_count": null_count,
            "null_pct": round(
                null_count / max(total_rows, 1) * 100,
                4
            ),
            "empty_count": empty_count,
            "empty_pct": round(
                empty_count / max(total_rows, 1) * 100,
                4
            )
        })

    return pd.DataFrame(results)


null_results = []

for view in VIEWS:
    print(f"QC: {view}")
    null_results.append(null_quality(view))

null_df = pd.concat(
    null_results,
    ignore_index=True
)

display(
    null_df[
        (null_df["null_count"] > 0) |
        (null_df["empty_count"] > 0)
    ]
    .sort_values(
        ["view", "null_pct"],
        ascending=[True, False]
    )
)

# atcStep3CdNm, atcStep4CdNm, st3SickSymNm 은 사용하지 않는다. 다만 코드만 사용한다!

QC: atc3_region_facil
QC: atc3_sick
QC: atc4_region_facil
QC: atc4_sick
QC: atc_master
QC: region_facil_stat


,view,column,data_type,null_count,null_pct,empty_count,empty_pct
11,atc3_region_facil,atcStep3CdNm,VARCHAR,495844,2.5440,0,0.0000
13,atc3_sick,atcStep3CdNm,VARCHAR,0,0.0000,577926,3.2963
17,atc3_sick,st3SickSymNm,VARCHAR,0,0.0000,108931,0.6213
31,atc4_region_facil,atcStep4CdNm,VARCHAR,436166,1.4831,0,0.0000
39,atc4_sick,atcStep4CdNm,VARCHAR,517190,2.0383,0,0.0000
33,atc4_sick,st3SickSymNm,VARCHAR,131127,0.5168,0,0.0000


In [29]:
def numeric_quality(view_name):

    columns = get_columns(view_name)

    numeric_columns = columns[
        columns["data_type"].str.upper().str.contains(
            "INTEGER|BIGINT|SMALLINT|TINYINT|HUGEINT|DECIMAL|DOUBLE|FLOAT"
        )
    ]

    if len(numeric_columns) == 0:
        return pd.DataFrame()

    expressions = []

    for _, row in numeric_columns.iterrows():

        col = row["column_name"]
        qcol = '"' + col.replace('"', '""') + '"'

        expressions.extend([
            f'MIN({qcol}) AS "{col}__min"',
            f'MAX({qcol}) AS "{col}__max"',
            f'AVG({qcol}) AS "{col}__avg"',

            f"""
            COUNT(*) FILTER (
                WHERE {qcol} < 0
            ) AS "{col}__negative"
            """,

            f"""
            COUNT(*) FILTER (
                WHERE {qcol} = 0
            ) AS "{col}__zero"
            """
        ])

    sql = f"""
        SELECT
            {','.join(expressions)}
        FROM "{view_name}"
    """

    raw = con.execute(sql).fetchdf()

    results = []

    for _, row in numeric_columns.iterrows():

        col = row["column_name"]

        results.append({
            "view": view_name,
            "column": col,
            "data_type": row["data_type"],
            "min": raw.iloc[0][f"{col}__min"],
            "max": raw.iloc[0][f"{col}__max"],
            "mean": raw.iloc[0][f"{col}__avg"],
            "negative_count": int(
                raw.iloc[0][f"{col}__negative"]
            ),
            "zero_count": int(
                raw.iloc[0][f"{col}__zero"]
            )
        })

    return pd.DataFrame(results)

numeric_results = []

for view in VIEWS:
    print(f"Numeric QC: {view}")

    result = numeric_quality(view)

    if not result.empty:
        numeric_results.append(result)

numeric_df = pd.concat(
    numeric_results,
    ignore_index=True
)

display(numeric_df)

Numeric QC: atc3_region_facil
Numeric QC: atc3_sick
Numeric QC: atc4_region_facil
Numeric QC: atc4_sick
Numeric QC: atc_master
Numeric QC: region_facil_stat


,view,column,data_type,min,max,mean,negative_count,zero_count
0,atc3_region_facil,totUseQty,DOUBLE,0.0,1.395778e+07,1.626758e+04,0,52207
1,atc3_region_facil,msupUseAmt,DOUBLE,0.0,1.323030e+10,7.097913e+06,0,8522
2,atc3_sick,totUseQty,BIGINT,0.0,1.512420e+08,1.500752e+04,0,56650
3,atc3_sick,msupUseAmt,BIGINT,0.0,6.164954e+10,6.846672e+06,0,1165
4,atc4_region_facil,totUseQty,DOUBLE,0.0,1.395770e+07,1.026053e+04,0,76409
5,atc4_region_facil,msupUseAmt,DOUBLE,0.0,8.610020e+09,4.442909e+06,0,11678
6,atc4_sick,totUseQty,DOUBLE,0.0,1.123388e+08,9.861909e+03,0,87141
7,atc4_sick,msupUseAmt,DOUBLE,0.0,4.492054e+10,4.453061e+06,0,1349
8,region_facil_stat,population,BIGINT,371895.0,1.358943e+07,3.037401e+06,0,0
9,region_facil_stat,pharm,BIGINT,146.0,6.008000e+03,1.496971e+03,0,0


In [19]:
diagym_views = []

for view in VIEWS:

    columns = get_columns(view)["column_name"].tolist()

    if "diagYm" in columns:
        diagym_views.append(view)

print("diagYm 포함 VIEW:")
print(diagym_views)

for view in diagym_views:

    print(f"\n[{view}] invalid diagYm")

    result = con.execute(f"""
        SELECT
            diagYm,
            COUNT(*) AS row_count
        FROM "{view}"
        WHERE diagYm IS NOT NULL
          AND NOT regexp_matches(
              CAST(diagYm AS VARCHAR),
              '^[0-9]{{6}}$'
          )
        GROUP BY diagYm
        ORDER BY row_count DESC
    """).fetchdf()

    display(result)


diagYm 포함 VIEW:
['atc3_region_facil', 'atc3_sick', 'atc4_region_facil', 'atc4_sick', 'region_facil_stat']

[atc3_region_facil] invalid diagYm


,diagYm,row_count



[atc3_sick] invalid diagYm


,diagYm,row_count



[atc4_region_facil] invalid diagYm


,diagYm,row_count



[atc4_sick] invalid diagYm


,diagYm,row_count



[region_facil_stat] invalid diagYm


,diagYm,row_count
0,2022,17
1,2021,17


In [20]:
date_range_df = []

for view in diagym_views:

    result = con.execute(f"""
        SELECT
            MIN(diagYm) AS min_diagYm,
            MAX(diagYm) AS max_diagYm,
            COUNT(DISTINCT diagYm) AS month_count
        FROM "{view}"
    """).fetchone()

    date_range_df.append({
        "view": view,
        "min_diagYm": result[0],
        "max_diagYm": result[1],
        "month_count": result[2]
    })

display(pd.DataFrame(date_range_df))

,view,min_diagYm,max_diagYm,month_count
0,atc3_region_facil,202001,202212,36
1,atc3_sick,202001,202212,36
2,atc4_region_facil,202001,202212,36
3,atc4_sick,202001,202212,36
4,region_facil_stat,2021,2022,2


In [ ]:
# import duckdb

# con = duckdb.connect("healthcare.duckdb")

# VIEWS = ['atc3_region_facil','atc3_sick','atc4_region_facil','atc4_sick']

def check_full_duplicates(con, view_name):
    # 해당 VIEW의 컬럼 목록 조회
    columns = (
        con.execute(f"""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'main'
              AND table_name = '{view_name}'
            ORDER BY ordinal_position
        """)
        .fetchdf()["column_name"]
        .tolist()
    )

    if not columns:
        raise ValueError(f"컬럼을 찾을 수 없습니다: {view_name}")

    # 컬럼명을 안전하게 SQL identifier로 처리
    hash_args = ", ".join(
        f'"{col.replace(chr(34), chr(34) * 2)}"'
        for col in columns
    )

    

    sql = f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash({hash_args})) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash({hash_args})) AS duplicate_rows
        FROM "{view_name}"
        -- WHERE DIAGYM = '202201';
    """

    print(sql)


    return con.execute(sql).fetchdf()


In [20]:


duplicate_results = {}

for view in VIEWS:
    print(f"Checking: {view}")

    duplicate_results[view] = check_full_duplicates(con, view)

    display(
        duplicate_results[view].assign(view_name=view)
    )

    

Checking: atc3_sick

        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash("atcStep3Cd", "atcStep3CdNm", "diagYm", "insupTpCd", "st3SickSym", "st3SickSymNm", "totUseQty", "msupUseAmt")) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash("atcStep3Cd", "atcStep3CdNm", "diagYm", "insupTpCd", "st3SickSym", "st3SickSymNm", "totUseQty", "msupUseAmt")) AS duplicate_rows
        FROM "atc3_sick"
        -- WHERE DIAGYM = '202201';
    


,total_rows,unique_rows,duplicate_rows,view_name
0,14364479,14364479,0,atc3_sick


Checking: atc4_sick

        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash("diagYm", "st3SickSymNm", "atcStep4Cd", "insupTpCd", "totUseQty", "msupUseAmt", "st3SickSym", "atcStep4CdNm")) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash("diagYm", "st3SickSymNm", "atcStep4Cd", "insupTpCd", "totUseQty", "msupUseAmt", "st3SickSym", "atcStep4CdNm")) AS duplicate_rows
        FROM "atc4_sick"
        -- WHERE DIAGYM = '202201';
    


,total_rows,unique_rows,duplicate_rows,view_name
0,20635895,20635895,0,atc4_sick


Checking: atc3_region_facil

        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash("diagYm", "atcStep2Cd", "atcStep3Cd", "regionStep2Cd", "regionStep1CdNm", "regionStep2CdNm", "insupTpCd", "regionStep1Cd", "totUseQty", "msupUseAmt", "medInstType", "atcStep3CdNm")) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash("diagYm", "atcStep2Cd", "atcStep3Cd", "regionStep2Cd", "regionStep1CdNm", "regionStep2CdNm", "insupTpCd", "regionStep1Cd", "totUseQty", "msupUseAmt", "medInstType", "atcStep3CdNm")) AS duplicate_rows
        FROM "atc3_region_facil"
        -- WHERE DIAGYM = '202201';
    


...

,total_rows,unique_rows,duplicate_rows,view_name
0,25748443,25748443,0,atc4_region_facil


Checking: region_facil_stat

        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash("diagYm", "sidoNm", "population", "pharm", "advGenHosp", "genHosp", "hosp", "longHosp", "clinic", "dentalClinic", "orientalClinic")) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash("diagYm", "sidoNm", "population", "pharm", "advGenHosp", "genHosp", "hosp", "longHosp", "clinic", "dentalClinic", "orientalClinic")) AS duplicate_rows
        FROM "region_facil_stat"
        -- WHERE DIAGYM = '202201';
    


,total_rows,unique_rows,duplicate_rows,view_name
0,34,34,0,region_facil_stat


Checking: atc_master

        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash("atc_code", "atc_name", "strength", "uom", "adm_r", "note")) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash("atc_code", "atc_name", "strength", "uom", "adm_r", "note")) AS duplicate_rows
        FROM "atc_master"
        -- WHERE DIAGYM = '202201';
    


,total_rows,unique_rows,duplicate_rows,view_name
0,7536,7536,0,atc_master


In [ ]:


duplicate_results = {}

for view in VIEWS:
    if view != 'atc3_sick':  continue;
    print(f"Checking: {view}")

    duplicate_results[view] = check_full_duplicates(con, view)

    display(
        duplicate_results[view].assign(view_name=view)
    )

    

Checking: atc3_sick

        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT hash("atcStep3Cd", "atcStep3CdNm", "diagYm", "insupTpCd", "msupUseAmt", "st3SickSym", "st3SickSymNm", "totUseQty")) AS unique_rows,
            COUNT(*) - COUNT(DISTINCT hash("atcStep3Cd", "atcStep3CdNm", "diagYm", "insupTpCd", "msupUseAmt", "st3SickSym", "st3SickSymNm", "totUseQty")) AS duplicate_rows
        FROM "atc3_sick"
        WHERE DIAGYM = '202201';
    


,total_rows,unique_rows,duplicate_rows,view_name
0,399021,399021,0,atc3_sick


In [ ]:
import duckdb

con = duckdb.connect("healthcare.duckdb")

def exact_duplicate_summary(con, view_name):
    columns = (
        con.execute(f"""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'main'
              AND table_name = '{view_name}'
            ORDER BY ordinal_position
        """)
        .fetchdf()["column_name"]
        .tolist()
    )

    if not columns:
        raise ValueError(f"컬럼을 찾을 수 없습니다: {view_name}")

    quoted_columns = ", ".join(
        f'"{c.replace(chr(34), chr(34) * 2)}"'
        for c in columns
    )

    sql = f"""
        WITH duplicate_groups AS (
            SELECT
                {quoted_columns},
                COUNT(*) AS group_count
            FROM "{view_name}"
            WHERE DIAGYM = '202201'
            GROUP BY {quoted_columns}
            HAVING COUNT(*) > 1
            
        )
        SELECT
            COUNT(*) AS total_groups,
            COALESCE(SUM(group_count - 1), 0) AS duplicate_rows,
            COALESCE(MAX(group_count), 0) AS max_repeat_count
        FROM duplicate_groups
        ;
    """

    return con.execute(sql).fetchdf()



VIEWS = ['atc3_region_facil','atc3_sick','atc4_region_facil','atc4_sick','mefi_sick']
for view in VIEWS:
    print(f"Checking exact duplicates: {view}")
    display(exact_duplicate_summary(con, view))
    break

Checking exact duplicates: atc3_region_facil


,total_groups,duplicate_rows,max_repeat_count
0,70476,70476.0,2


In [10]:
def exact_duplicate_summary(con, view_name):
    sql = f"""
        WITH duplicate_groups AS (
    SELECT
        diagYm,
        atcStep2Cd,
        atcStep3Cd,
        regionStep2Cd,
        regionStep1Cd,
        insupTpCd,
        medInstType,
        COUNT(*) AS group_count,
        SUM(totUseQty) AS sum_use_qty,
        SUM(msupUseAmt) AS sum_use_amt,
        MIN(totUseQty) AS min_use_qty,
        MAX(totUseQty) AS max_use_qty,
        MIN(msupUseAmt) AS min_use_amt,
        MAX(msupUseAmt) AS max_use_amt
    FROM atc3_region_facil
    WHERE diagYm = '202201'
    GROUP BY
        diagYm,
        atcStep2Cd,
        atcStep3Cd,
        regionStep2Cd,
        regionStep1Cd,
        insupTpCd,
        medInstType
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS duplicate_groups,

    SUM(group_count - 1) AS extra_rows,

    SUM(
        CASE
            WHEN min_use_amt = max_use_amt
            THEN (group_count - 1) * min_use_amt
            ELSE 0
        END
    ) AS potentially_duplicated_amount,

    SUM(
        CASE
            WHEN min_use_amt <> max_use_amt
            THEN 1
            ELSE 0
        END
    ) AS groups_with_different_amount,

    SUM(group_count) AS rows_in_duplicate_groups
FROM duplicate_groups;
    """

    return con.execute(sql).fetchdf()

for view in VIEWS:
    print(f"Checking exact duplicates: {view}")

    result = exact_duplicate_summary(con, view)
    display(result)
    break;

#     	view	row_count
# 0	atc4_sick	25374158
# 1	atc3_sick	17532309
# 2	cmpn_sick	47719163
# 3	mefi_sick	12634246


Checking exact duplicates: atc3_region_facil


,duplicate_groups,extra_rows,potentially_duplicated_amount,groups_with_different_amount,rows_in_duplicate_groups
0,200247,200247.0,2.240779e+11,129757.0,400494.0


In [12]:
con.execute("""SELECT
    COUNT(*) AS total_rows,
    SUM(msupUseAmt) AS total_use_amt,
    SUM(msupUseAmt) / 1e12 AS total_trillion
FROM atc3_region_facil
WHERE diagYm = '202201';""").df()

,total_rows,total_use_amt,total_trillion
0,549559,3.899617e+12,3.899617


In [16]:
con.execute("""
WITH duplicate_groups AS (
    SELECT
        diagYm,
        atcStep2Cd,
        atcStep3Cd,
        regionStep2Cd,
        regionStep1Cd,
        insupTpCd,
        medInstType,
        tOTUseQty,
        msupUseAmt
    FROM atc3_region_facil
    WHERE diagYm = '202201'
    GROUP BY
        diagYm,
        atcStep2Cd,
        atcStep3Cd,
        regionStep2Cd,
        regionStep1Cd,
        insupTpCd,
        medInstType,
        tOTUseQty,
        msupUseAmt
    HAVING COUNT(*) > 1
)
SELECT
    a.diagYm,
    a.atcStep2Cd,
    a.atcStep3Cd,
    a.regionStep1Cd,
    a.regionStep2Cd,
    a.insupTpCd,
    a.medInstType,
    a.totUseQty,
    a.msupUseAmt
FROM atc3_region_facil a
JOIN duplicate_groups d
  ON a.diagYm = d.diagYm
 AND a.atcStep2Cd = d.atcStep2Cd
 AND a.atcStep3Cd = d.atcStep3Cd
 AND a.regionStep2Cd = d.regionStep2Cd
 AND a.regionStep1Cd = d.regionStep1Cd
 AND a.insupTpCd = d.insupTpCd
 AND a.medInstType = d.medInstType
 AND a.totUseQty = d.totUseQty
    AND a.msupUseAmt = d.msupUseAmt
WHERE a.diagYm = '202201'
order by a.diagYm, a.atcStep2Cd, a.atcStep3Cd, a.regionStep1Cd, a.regionStep2Cd, a.insupTpCd, a.medInstType
LIMIT 50;
""").df()

,diagYm,atcStep2Cd,atcStep3Cd,regionStep1Cd,regionStep2Cd,insupTpCd,medInstType,totUseQty,msupUseAmt
0,202201,A02,A02A,01,110020,5,상급종합병원,2278.0,105332.0
1,202201,A02,A02A,01,110020,5,상급종합병원,2278.0,105332.0
2,202201,A02,A02A,11,220100,5,종합병원,603.0,16502.0
3,202201,A02,A02A,11,220100,5,종합병원,603.0,16502.0
4,202201,A02,A02A,11,230004,4,종합병원,581.0,29569.0
5,202201,A02,A02A,11,230004,4,종합병원,581.0,29569.0
6,202201,A02,A02A,11,230004,5,종합병원,602.0,9554.0
7,202201,A02,A02A,11,230004,5,종합병원,602.0,9554.0
8,202201,A02,A02A,11,312100,5,종합병원,4.0,40.0
9,202201,A02,A02A,11,312100,5,종합병원,4.0,40.0


In [ ]:
# print("=" * 100)
# print("DATA QUALITY SUMMARY")
# print("=" * 100)

# print("\n[1] ROW COUNT")
# display(row_counts)

# print("\n[2] NULL / EMPTY")
# display(
#     null_df[
#         (null_df["null_count"] > 0) |
#         (null_df["empty_count"] > 0)
#     ]
#     .sort_values(
#         ["view", "null_pct"],
#         ascending=[True, False]
#     )
# )

# print("\n[3] NUMERIC")
# display(numeric_df)

# print("\n[4] DATE RANGE")
# display(pd.DataFrame(date_range_df))

# print("\n[5] DUPLICATE")
# display(duplicate_df)

In [ ]:
#  for file in files:
#     schema = con.execute(f"""
#         DESCRIBE SELECT *
#         FROM read_parquet('{file}')
#     """).fetchdf()

#     display(schema)
#  # 수치형 후보
#     numeric_cols = []

#     for col in schema["column_name"]:

#         result = con.execute(f"""
#             SELECT
#                 COUNT(*) AS n,
#                 COUNT(
#                     TRY_CAST("{col}" AS DOUBLE)
#                 ) AS numeric_n
#             FROM read_parquet('{file}')
#         """).fetchone()

#         n, numeric_n = result

#         if n > 0 and numeric_n / n >= 0.95:
#             numeric_cols.append(col)

#     print("\n[NUMERIC COLUMNS]")
#     print(numeric_cols)

#     # 통계
#     for col in numeric_cols:

#         stats = con.execute(f"""
#             SELECT
#                 MIN(TRY_CAST("{col}" AS DOUBLE)) AS min,
#                 QUANTILE_CONT(
#                     TRY_CAST("{col}" AS DOUBLE), 0.25
#                 ) AS q1,
#                 MEDIAN(
#                     TRY_CAST("{col}" AS DOUBLE)
#                 ) AS median,
#                 AVG(
#                     TRY_CAST("{col}" AS DOUBLE)
#                 ) AS mean,
#                 QUANTILE_CONT(
#                     TRY_CAST("{col}" AS DOUBLE), 0.75
#                 ) AS q3,
#                 MAX(
#                     TRY_CAST("{col}" AS DOUBLE)
#                 ) AS max
#             FROM read_parquet('{file}')
#         """).fetchdf()

#         print(f"\n[{col}]")
#         display(stats)

#         values = con.execute(f"""
#             SELECT TRY_CAST("{col}" AS DOUBLE) AS value
#             FROM read_parquet('{file}')
#             WHERE TRY_CAST("{col}" AS DOUBLE) IS NOT NULL
#         """).fetchdf()

#         plt.figure(figsize=(9, 3))
#         plt.boxplot(values["value"], vert=False)
#         plt.title(f"{file.name} — {col}")
#         plt.xlabel(col)
#         plt.show()

# date_range_df = []

# for view in diagym_views:

#     result = con.execute(f"""
#         SELECT
#             MIN(diagYm) AS min_diagYm,
#             MAX(diagYm) AS max_diagYm,
#             COUNT(DISTINCT diagYm) AS month_count
#         FROM "{view}"
#     """).fetchone()

#     date_range_df.append({
#         "view": view,
#         "min_diagYm": result[0],
#         "max_diagYm": result[1],
#         "month_count": result[2]
#     })

# display(pd.DataFrame(date_range_df))

NameError: name 'files' is not defined

## 요청사항 점검

DuckDB에 현재 생성된 뷰/테이블의 구조, 기간, 대표 코드값, 기관종별/보험종별 값, ATC master 및 성장분석 결과를 점검한다.

In [4]:
import re


def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


catalog = con.execute("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchdf()

available = set(catalog["table_name"].tolist())

print("[현재 main 스키마 객체]")
display(catalog)


# 1~3. 요청된 테이블의 DESCRIBE
requested_objects = [
    "atc3_region_facil",
    "atc4_region_facil",
    "region_facil_stat",
    "atc4_sick",
    "atc_master"
]

print("[1~3] DESCRIBE")
for object_name in requested_objects:
    print(f"\n--- {object_name} ---")
    if object_name in available:
        display(con.execute(
            f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
        ).df())
    else:
        print("객체가 존재하지 않습니다.")


# 4. diagYm의 실제 최소/최대값
print("[4] MIN/MAX diagYm")
diagym_objects = [
    object_name for object_name in requested_objects
    if object_name in available
    and "diagYm" in con.execute(
        f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
    ).df()["column_name"].tolist()
]

diagym_summary = []
for object_name in diagym_objects:
    result = con.execute(f"""
        SELECT
            '{object_name}' AS object_name,
            MIN(CAST(diagYm AS VARCHAR)) AS min_diagYm,
            MAX(CAST(diagYm AS VARCHAR)) AS max_diagYm,
            COUNT(DISTINCT diagYm) AS distinct_diagYm
        FROM {quote_identifier(object_name)}
    """).df()
    diagym_summary.append(result)

display(pd.concat(diagym_summary, ignore_index=True))


# 5. 지역 컬럼의 실제 값 10개
print("[5] 지역 컬럼 실제 값 (최대 10개)")
region_specs = {
    "atc3_region_facil": ["regionStep1Cd", "regionStep1CdNm", "regionStep2Cd", "regionStep2CdNm"],
    "atc4_region_facil": ["regionStep1Cd", "regionStep1CdNm", "regionStep2Cd", "regionStep2CdNm"],
    "region_facil_stat": ["sidoNm"],
}

for object_name, candidate_columns in region_specs.items():
    if object_name not in available:
        continue
    columns = con.execute(
        f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
    ).df()["column_name"].tolist()
    for column_name in candidate_columns:
        if column_name not in columns:
            continue
        print(f"\n--- {object_name}.{column_name} ---")
        display(con.execute(f"""
            SELECT DISTINCT {quote_identifier(column_name)} AS value
            FROM {quote_identifier(object_name)}
            WHERE {quote_identifier(column_name)} IS NOT NULL
            ORDER BY value
            LIMIT 10
        """).df())


# 6. 의료기관종별 컬럼의 실제 값 전체
print("[6] 의료기관종별 값 전체")
for object_name in ["atc3_region_facil", "atc4_region_facil"]:
    if object_name not in available:
        continue
    columns = con.execute(
        f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
    ).df()["column_name"].tolist()
    if "medInstType" in columns:
        print(f"\n--- {object_name}.medInstType ---")
        display(con.execute(f"""
            SELECT DISTINCT medInstType
            FROM {quote_identifier(object_name)}
            ORDER BY medInstType
        """).df())


# 7. insupTpCd 실제 값
print("[7] insupTpCd 실제 값")
for object_name in requested_objects:
    if object_name not in available:
        continue
    columns = con.execute(
        f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
    ).df()["column_name"].tolist()
    if "insupTpCd" in columns:
        print(f"\n--- {object_name}.insupTpCd ---")
        display(con.execute(f"""
            SELECT DISTINCT insupTpCd
            FROM {quote_identifier(object_name)}
            ORDER BY insupTpCd
        """).df())


# 8. ATC master 객체명과 DESCRIBE
print("[8] ATC master 객체와 DESCRIBE")
atc_master_objects = sorted(
    object_name for object_name in available
    if "atc" in object_name.lower() and "master" in object_name.lower()
)
if not atc_master_objects:
    print("ATC master 객체를 찾지 못했습니다.")
for object_name in atc_master_objects:
    print(f"\n--- {object_name} ---")
    display(con.execute(
        f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
    ).df())


# 9. 성장분석 결과 후보와 DESCRIBE
print("[9] 성장분석 결과 후보")
growth_keywords = ("candidate", "growth", "market", "trend", "result")
growth_objects = sorted(
    object_name for object_name in available
    if any(keyword in object_name.lower() for keyword in growth_keywords)
)
if not growth_objects:
    print("성장분석 결과 후보 객체를 찾지 못했습니다.")
for object_name in growth_objects:
    print(f"\n--- {object_name} ---")
    display(con.execute(
        f"DESCRIBE SELECT * FROM {quote_identifier(object_name)}"
    ).df())


# 10. 주요 객체와 발견된 master/성장 결과의 대략적인 row count
print("[10] 대략적인 row count")
count_objects = sorted(set(
    object_name for object_name in requested_objects + atc_master_objects + growth_objects
    if object_name in available
))
row_counts = []
for object_name in count_objects:
    count = con.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(object_name)}"
    ).fetchone()[0]
    row_counts.append({"object_name": object_name, "row_count": count})

display(pd.DataFrame(row_counts))

[현재 main 스키마 객체]


,table_name,table_type
0,atc3_region_facil,VIEW
1,atc3_sick,VIEW
2,atc4_region_facil,VIEW
3,atc4_sick,VIEW
4,atc_master,VIEW
5,region_facil_stat,VIEW


[1~3] DESCRIBE

--- atc3_region_facil ---


,column_name,column_type,null,key,default,extra
0,diagYm,VARCHAR,YES,None,None,None
1,atcStep2Cd,VARCHAR,YES,None,None,None
2,atcStep3Cd,VARCHAR,YES,None,None,None
3,regionStep2Cd,VARCHAR,YES,None,None,None
4,regionStep1CdNm,VARCHAR,YES,None,None,None
5,regionStep2CdNm,VARCHAR,YES,None,None,None
6,insupTpCd,VARCHAR,YES,None,None,None
7,regionStep1Cd,VARCHAR,YES,None,None,None
8,totUseQty,DOUBLE,YES,None,None,None
9,msupUseAmt,DOUBLE,YES,None,None,None



--- atc4_region_facil ---


...

,insupTpCd
0,4
1,5
2,7


[8] ATC master 객체와 DESCRIBE

--- atc_master ---


,column_name,column_type,null,key,default,extra
0,atc_code,VARCHAR,YES,None,None,None
1,atc_name,VARCHAR,YES,None,None,None
2,strength,VARCHAR,YES,None,None,None
3,uom,VARCHAR,YES,None,None,None
4,adm_r,VARCHAR,YES,None,None,None
5,note,VARCHAR,YES,None,None,None


[9] 성장분석 결과 후보
성장분석 결과 후보 객체를 찾지 못했습니다.
[10] 대략적인 row count


,object_name,row_count
0,atc3_region_facil,19490504
1,atc4_region_facil,29409519
2,atc4_sick,25374158
3,atc_master,7536
4,region_facil_stat,34
